In [1]:
import copy
import os
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import sys
os.environ['DISPLAY'] = ':1'
sys.path += [".."]
sys.path += [str(pathlib.Path(os.getcwd()) / '..' / "diff_co_mpc")]
from pydrake.geometry import (
    ClippingRange,
    ColorRenderCamera,
    DepthRange,
    DepthRenderCamera,
    MakeRenderEngineVtk,
    RenderCameraCore,
    RenderEngineVtkParams,
    RenderLabel,
    
    Role,
    StartMeshcat,
)
from pydrake.perception import DepthImageToPointCloud
from pydrake.math import RigidTransform, RollPitchYaw
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.multibody.tree import BodyIndex
from pydrake.systems.analysis import Simulator
from pydrake.systems.framework import DiagramBuilder
from pydrake.systems.sensors import (
    CameraInfo,
    RgbdSensor,
)
from pydrake.visualization import (
    AddDefaultVisualization,
    
    ColorizeDepthImage,
    ColorizeLabelImage,
)
import pydrake.geometry as pydgeo
from pydrake.all import (
    UnitInertia,
    SpatialInertia,
    MeshcatPointCloudVisualizer,
    LeafSystem,
    FixedOffsetFrame,
    AbstractValue, BaseField,PixelType,LightParameter,Rgba, DrakeLcm, LcmSubscriberSystem,LcmImageArrayToImages,LcmInterfaceSystem,Value,PointCloud,ImageDepth16U,ImageDepth32F,ImageToLcmImageArrayT,LcmPublisherSystem,ImageRgba8U,Image
)
from diff_co_mpc.diff_co_lcm import lcmt_pose
import time
from drake import lcmt_robot_state, lcmt_point_cloud, lcmt_image, lcmt_image_array
import copy
import utils
from mpc.optimisation import make_system
from mpc.plant import plant_from_yaml
try:
    meshcat
    print("meshcat already defined")
except NameError:
    meshcat = StartMeshcat()

TEMP_FOLDER = pathlib.Path(os.getcwd()) / '..' / "temp"
TEMP_FOLDER.mkdir(exist_ok=True)
CODEGEN_FOLDER = TEMP_FOLDER / "codegen"
CODEGEN_FOLDER.mkdir(exist_ok=True, parents=True)
yaml_path = pathlib.Path('..') / 'config'/ "plant_definition.yaml"

system = make_system(yaml_path, meshcat, CODEGEN_FOLDER, camera = False)
complete_plant = system["complete_plant"]
robots = system["robots"]
viz_helper = system["visualization_helper"]
carried_object = system["carried_object"]
yaml = system["yaml"]

Unknown module.None():0 INFO -- Meshcat listening for connections at http://localhost:7000


Model instance index: ModelInstanceIndex(9)
Model instance index: ModelInstanceIndex(2)
Model instance index: ModelInstanceIndex(10)
Model instance index: ModelInstanceIndex(2)
Compiling  IK_obj
gcc -Ofast -fopenmp -march=native -shared /tmp/tmpumd8shrp/IK_obj.c -o /tmp/tmp5dy_eut6/IK_obj.so
stdout:
 
stderr:
 
IK_obj  compiled successfully
Compiling  IK_EE
gcc -Ofast -fopenmp -march=native -shared /tmp/tmp_pl0ghme/IK_EE.c -o /tmp/tmp5dy_eut6/IK_EE.so
stdout:
 
stderr:
 
IK_EE  compiled successfully
Compiling  IK_obj
gcc -Ofast -fopenmp -march=native -shared /tmp/tmpde9kf4sz/IK_obj.c -o /tmp/tmpgnjwr3ve/IK_obj.so
stdout:
 
stderr:
 
IK_obj  compiled successfully
Compiling  IK_EE
gcc -Ofast -fopenmp -march=native -shared /tmp/tmpeqnqf164/IK_EE.c -o /tmp/tmpgnjwr3ve/IK_EE.so
stdout:
 
stderr:
 
IK_EE  compiled successfully


In [2]:
class PlantDiagramWrap(LeafSystem):
    def __init__(self, yaml_path, meshcat, point_cloud_visualizer):
        super().__init__()
        self.plant_info = plant_from_yaml(yaml_path, camera = True, meshcat = meshcat, point_cloud_visualizer=point_cloud_visualizer)
        self.plants_info = self.plant_info['plants_info']
        self.robot_1_model_instance = self.plants_info['robot_opt_tuple'][0].viz_model_instance
        self.robot_2_model_instance = self.plants_info['robot_opt_tuple'][1].viz_model_instance
        self.cameras = self.plants_info['camera']
        self.diagram = self.plants_info['diagram']
        self.plant = self.plants_info['plant']
        self.visualizer = self.diagram.GetSubsystemByName('meshcat_visualizer(illustration)')
        # self.point_cloud_visualizer = self.diagram.GetSubsystemByName('point_cloud_visualizer')
        self.depth_to_point_cloud = [camera._depth_to_point_cloud for camera in self.cameras]
        self.sensor = [camera._sensor for camera in self.cameras]
        self.context = self.diagram.CreateDefaultContext()
        self.context_plant = self.plant.GetMyContextFromRoot(self.context)
        self.viz_context = self.visualizer.GetMyContextFromRoot(self.context)
        # self.pc_visualizer_context = self.point_cloud_visualizer.GetMyContextFromRoot(self.context)
        self.depth_to_point_cloud_context = [depth_to_point_cloud.GetMyContextFromRoot(self.context) for depth_to_point_cloud in self.depth_to_point_cloud]
        self.sensor_context = [sensor.GetMyContextFromRoot(self.context) for sensor in self.sensor]
        # self.DeclareForcedPublishEvent(self.publish)
        self.robot_1_position_port = self.DeclareVectorInputPort(name = "robot_1_position", size = 9)
        self.robot_2_position_port = self.DeclareVectorInputPort(name = "robot_2_position", size = 9)
        self.camera_pose_port = self.DeclareAbstractInputPort(name="in",
                                      model_value=Value(RigidTransform()))
        self.robot_1_lcm_position_output_port = self.DeclareAbstractOutputPort(name="robot_1_position_lcm",
                                      alloc=lambda:Value(lcmt_robot_state()),
                                      calc = self.calc_robot_1_lcm_position)
        self.robot_2_lcm_position_output_port = self.DeclareAbstractOutputPort(name="robot_2_position_lcm",
                                      alloc=lambda:Value(lcmt_robot_state()),
                                      calc = self.calc_robot_2_lcm_position)
        self.camera_pose_lcm_output_port = self.DeclareAbstractOutputPort(name="camera_pose_lcm",
                                      alloc=lambda:Value(lcmt_pose()),
                                      calc = self.calc_camera_pose_lcm)
        self.camera_pose_lcm_output_port_2 = self.DeclareAbstractOutputPort(name="camera_pose_lcm_2",
                                      alloc=lambda:Value(lcmt_pose()),
                                      calc = self.calc_camera_pose_lcm_2)
        self.point_cloud_output_port = self.DeclareAbstractOutputPort(
            name="out",
            alloc=lambda: Value(PointCloud()),
            calc=self.calc_point_cloud)
        self.depth_output_port = self.DeclareAbstractOutputPort(
            name="depth_out",
            # alloc=lambda: Value(ImageDepth16U()),
            alloc=lambda: Value(ImageDepth32F()),
            calc=self.calc_depth)
        self.color_output_port = self.DeclareAbstractOutputPort(
            name="color_out",
            alloc=lambda: Value(ImageRgba8U()),
            calc=self.calc_color)
        self.label_output_port = self.DeclareAbstractOutputPort(
            name="label_out",
            alloc=lambda: Value(ImageLabel16I()),
            calc=self.calc_label)
        
        self.point_cloud_output_port_2 = self.DeclareAbstractOutputPort(
            name="out_2",
            alloc=lambda: Value(PointCloud()),
            calc=self.calc_point_cloud_2)
        self.depth_output_port_2 = self.DeclareAbstractOutputPort(
            name="depth_out_2",
            # alloc=lambda: Value(ImageDepth16U()),
            alloc=lambda: Value(ImageDepth32F()),
            calc=self.calc_depth_2)
        self.color_output_port_2 = self.DeclareAbstractOutputPort(
            name="color_out_2",
            alloc=lambda: Value(ImageRgba8U()),
            calc=self.calc_color_2)
        self.label_output_port_2 = self.DeclareAbstractOutputPort(
            name="label_out_2",
            alloc=lambda: Value(ImageLabel16I()),
            calc=self.calc_label_2)
        
    def calc_robot_1_lcm_position(self, context, output):
        robot_1_position = self.robot_1_position_port.Eval(context)
        robot_2_position = self.robot_2_position_port.Eval(context)
        # camera_pose = self.camera_pose_port.Eval(context)
        self.plant.SetPositions(self.context_plant,self.robot_1_model_instance, robot_1_position)
        self.plant.SetPositions(self.context_plant,self.robot_2_model_instance, robot_2_position)
        # self.cameras[0].set_transform(self.context_plant,camera_pose)
        robot_1_lcm_position = lcmt_robot_state()
        robot_1_lcm_position.joint_name = ['joint_{}'.format(i) for i in range(9)]
        robot_1_lcm_position.num_joints = 9
        robot_1_lcm_position.joint_position = robot_1_position
        output.set_value(robot_1_lcm_position)
    def calc_camera_pose_lcm(self, context, output):
        camera_transform = self.camera_pose_port.Eval(context)
        self.cameras[0].set_transform(self.context_plant,camera_transform)
        pose_lcm = lcmt_pose()
        pose_lcm.pose = camera_transform.GetAsMatrix4().flatten()
        output.set_value(pose_lcm)
    def calc_camera_pose_lcm_2(self, context, output):        
        pose_lcm = lcmt_pose()
        # pose_lcm.pose = self.cameras[1].get_transform(self.context_plant).GetAsMatrix4().flatten()
        pose_lcm.pose = self.cameras[1]._frame.CalcPoseInWorld(self.context_plant).GetAsMatrix4().flatten()
        output.set_value(pose_lcm)
    def calc_robot_2_lcm_position(self, context, output):
        robot_1_position = self.robot_1_position_port.Eval(context)
        robot_2_position = self.robot_2_position_port.Eval(context)
        # camera_pose = self.camera_pose_port.Eval(context)
        self.plant.SetPositions(self.context_plant,self.robot_1_model_instance, robot_1_position)
        self.plant.SetPositions(self.context_plant,self.robot_2_model_instance, robot_2_position)
        # self.cameras.set_transform(self.context_plant,camera_pose)
        robot_2_lcm_position = lcmt_robot_state()
        robot_2_lcm_position.joint_name = ['joint_{}'.format(i) for i in range(7)] + [ "panda_finger_joint1","panda_finger_joint2"]
        robot_2_lcm_position.num_joints = 9
        robot_2_lcm_position.joint_position = robot_2_position
        output.set_value(robot_2_lcm_position)

    # def publish(self, context, ):
    #     robot_1_position = self.robot_1_position_port.Eval(context)
    #     robot_2_position = self.robot_2_position_port.Eval(context)
    #     camera_pose = self.camera_pose_port.Eval(context)
    #     self.plant.SetPositions(self.context_plant,self.robot_1_model_instance, robot_1_position)
    #     self.plant.SetPositions(self.context_plant,self.robot_2_model_instance, robot_2_position)
    #     self.camera.set_transform(self.context_plant,camera_pose)
        # self.visualizer.ForcedPublish(self.viz_context)
    def calc_point_cloud(self, context, output):
        output.set_value(self.depth_to_point_cloud[0].GetOutputPort('point_cloud').Eval(self.depth_to_point_cloud_context[0]))
    def calc_depth(self, context, output):
        # output.set_value(self.camera._sensor.GetOutputPort('depth_image_16u').Eval(self.sensor_context))
        output.set_value(self.cameras[0]._sensor.GetOutputPort('depth_image_32f').Eval(self.sensor_context[0]))
    def calc_color(self, context, output):        
        output.set_value(self.cameras[0]._sensor.GetOutputPort('color_image').Eval(self.sensor_context[0]))
    def calc_label(self, context, output):        
        output.set_value(self.cameras[0]._sensor.GetOutputPort('label_image').Eval(self.sensor_context[0]))

    def calc_point_cloud_2(self, context, output):
        output.set_value(self.depth_to_point_cloud[1].GetOutputPort('point_cloud').Eval(self.depth_to_point_cloud_context[1]))
    def calc_depth_2(self, context, output):
        # output.set_value(self.camera._sensor.GetOutputPort('depth_image_16u').Eval(self.sensor_context))
        output.set_value(self.cameras[1]._sensor.GetOutputPort('depth_image_32f').Eval(self.sensor_context[1]))
    def calc_color_2(self, context, output):        
        output.set_value(self.cameras[1]._sensor.GetOutputPort('color_image').Eval(self.sensor_context[1]))
    def calc_label_2(self, context, output):        
        output.set_value(self.cameras[1]._sensor.GetOutputPort('label_image').Eval(self.sensor_context[1]))
from pydrake.all import ImageLabel16I
        

In [ ]:
from diff_co_mpc.diff_co_lcm import lcmt_global_solve
from utils.math.BSpline import BSpline
import time
class TrajectoryTracker(LeafSystem):
    def __init__(self, ):
        super().__init__()
        # self.DeclareForcedPublishEvent(self.publish)
        self.trajectory_input_port = self.DeclareAbstractInputPort(name="in",
                                      model_value=Value(lcmt_global_solve()))
        self.bspline_1 = None
        self.bspline_2 = None
        self.t0 = -1
        self.output_1_port = self.DeclareVectorOutputPort("position_1",size= 9, calc =self.calc_1)
        self.output_2_port = self.DeclareVectorOutputPort("position_2",size= 9, calc =self.calc_2)
    def calc_1(self, context, output):
        t = time.perf_counter_ns()
        lcmt_trajetory = self.trajectory_input_port.Eval(context)
        if lcmt_trajetory.utime == 0:
            output.set_value(np.zeros(9))
            return
        print(t)
        # if not self.bspline_1:
        self.bspline_1 = BSpline(np.asarray(lcmt_trajetory.bspline_robot_1.control_points), 3)
        self.t0 = lcmt_trajetory.utime
        s = ((t - self.t0) / 1e9 / 10)%1
        q1 = self.bspline_1.evaluate(s)
        q1 = np.concatenate([q1,np.zeros(2)])
        
        output.set_value(q1)
        
        pass
    def calc_2(self, context, output):
        t = time.perf_counter_ns()
        lcmt_trajetory = self.trajectory_input_port.Eval(context)
        if lcmt_trajetory.utime == 0:
            output.set_value(np.zeros(9))
            return
        # if not self.bspline_2:
        self.bspline_2 = BSpline(np.asarray(lcmt_trajetory.bspline_robot_2.control_points), 3)
        s = ((t - self.t0) / 1e9 / 10)%1
        q2 = self.bspline_2.evaluate(s)
        
        q2 = np.concatenate([q2,np.zeros(2)])
        output.set_value(q2)
        print(s)
        pass
builder = DiagramBuilder()



p = PlantDiagramWrap(str(yaml_path), meshcat, True)

lcm = DrakeLcm()
builder.AddSystem(p)
lcm_sys = builder.AddSystem(LcmInterfaceSystem(lcm))
lcm_image_publisher = builder.AddSystem(LcmPublisherSystem.Make("simulation_depth_image",lcm =lcm_sys,lcm_type=lcmt_image_array,use_cpp_serializer=True))
lcm_image_publisher_2 = builder.AddSystem(LcmPublisherSystem.Make("simulation_depth_image_2",lcm =lcm_sys,lcm_type=lcmt_image_array,use_cpp_serializer=True))
robot_1_position_publisher = builder.AddSystem(LcmPublisherSystem.Make("robot_1_position",lcm =lcm_sys,lcm_type=lcmt_robot_state,use_cpp_serializer=False))
robot_2_position_publisher = builder.AddSystem(LcmPublisherSystem.Make("robot_2_position",lcm =lcm_sys,lcm_type=lcmt_robot_state,use_cpp_serializer=False))
camera_pose_publisher = builder.AddSystem(LcmPublisherSystem.Make("camera_pose",lcm =lcm_sys,lcm_type=lcmt_pose,use_cpp_serializer=False))
camera_pose_publisher_2 = builder.AddSystem(LcmPublisherSystem.Make("camera_pose_2",lcm =lcm_sys,lcm_type=lcmt_pose,use_cpp_serializer=False))
global_trajectory_subscriber = builder.AddSystem(LcmSubscriberSystem.Make("global_trajectory",lcm =lcm_sys,lcm_type=lcmt_global_solve,use_cpp_serializer=False))
trajectory_tracker = builder.AddSystem(TrajectoryTracker())
builder.Connect(global_trajectory_subscriber.get_output_port(),trajectory_tracker.trajectory_input_port)
builder.Connect(trajectory_tracker.output_1_port,p.robot_1_position_port)
builder.Connect(trajectory_tracker.output_2_port,p.robot_2_position_port)

image_to_lcm_array = builder.AddSystem(ImageToLcmImageArrayT())
image_to_lcm_array_2 = builder.AddSystem(ImageToLcmImageArrayT())
# image_to_lcm_array = builder.AddSystem(ImageToLcmImageArrayT('a','b','c'))
builder.Connect(p.depth_output_port,image_to_lcm_array.DeclareImageInputPort[PixelType.kDepth32F]('depth_image'))
builder.Connect(p.depth_output_port_2,image_to_lcm_array_2.DeclareImageInputPort[PixelType.kDepth32F]('depth_image'))
# builder.Connect(p.color_output_port,image_to_lcm_array.DeclareImageInputPort[PixelType.kRgba8U]('color_image'))
# builder.Connect(p.depth_output_port,image_to_lcm_array.depth_image_input_port())
# builder.Connect(p.color_output_port,image_to_lcm_array.color_image_input_port())
# builder.Connect(p.label_output_port,image_to_lcm_array.label_image_input_port())


builder.Connect(p.robot_1_lcm_position_output_port,robot_1_position_publisher.get_input_port())
builder.Connect(p.robot_2_lcm_position_output_port,robot_2_position_publisher.get_input_port())
builder.Connect(p.camera_pose_lcm_output_port,camera_pose_publisher.get_input_port())
builder.Connect(p.camera_pose_lcm_output_port_2,camera_pose_publisher_2.get_input_port())


builder.Connect(image_to_lcm_array.image_array_t_msg_output_port(),lcm_image_publisher.get_input_port())
builder.Connect(image_to_lcm_array_2.image_array_t_msg_output_port(),lcm_image_publisher_2.get_input_port())
# lcm_image_publisher.GetInputPort('lcm_message')
diag = builder.Build()
d_context = diag.CreateDefaultContext()
p_context = p.GetMyContextFromRoot(d_context)
simulator = Simulator(diag,d_context)
obstacle_positions= np.array([
                    [0.3,0.0,0.4],
                    [-0.2,-0.35,0.3],
                    [0.5,-0.25,0.6],
                    [-0.2,-0.35,0.3],
                    [0.2,-0.35,0.3],
                    [0.0,-0.1,0.05],
                    [0.0,-0.15,0.25],
                    ])
obstacle_positions= np.array([
                    [0.1,0.0,0.3],
                    [-0.1,-0.0,0.3],
                    [0.5,0.15,0.6],
                    [-0.2,-0.35,0.3],
                    [0.0,-0.0,0.6],
                    [0.0,-0.1,0.05],
                    [0.0,-0.15,6.25],
                    ])
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_1'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[0]]))
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_2'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[1]]))
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_3'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[2]]))
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_4'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[3]]))
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_5'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[4]]))
# asdfasdf
# robot_camera_depth_in_EE = np.array([[-0.00380074, -0.99995455, -0.00878683,  0.08132166],
#        [ 0.99953527, -0.00406475,  0.03022601, -0.06282357],
#        [-0.03026036, -0.00866787,  0.99950444, -0.073011  ],
#        [ 0.        ,  0.        ,  0.        ,  1.        ]])
# p.camera_pose_port.FixValue(p_context, RigidTransform(p=[0,2.,.5],rpy = RollPitchYaw(np.pi/2,np.pi,0)))
p.camera_pose_port.FixValue(p_context, RigidTransform(p=[0.,-.8*2,.25],rpy = RollPitchYaw(-np.pi/2,np.pi*0,0)))
t0 = time.perf_counter_ns()/1e9
# d_context.SetTime(t0)
t = 0
# obstacle_positions[0,2] += 1000
# obstacle_positions[1,2] += 1000
obstacle_positions[2,2] += 1000
obstacle_positions[3,2] += 1000
obstacle_positions[4,2] += 1000
# obstacle_positions[5,2] += 1000
# obstacle_positions[6,2] += 1000
q2 = [0,np.pi/2+0.3*np.sin(t*0.5),0,0,0,0,0.]
from pydrake.all import RotationMatrix
while True:
    t = time.perf_counter_ns()/1e9
    q1 = [np.pi/4 + 0.3*np.sin(t*1)*0-np.pi,np.pi/8,0,np.pi/3,-np.pi,np.pi/2,0.,0,0]
    q2 = [0,np.pi/2+0.3*np.sin(t*1)*0,0,0,0,0,0.,0,0,]
    q1 = np.array(q1)*0
    q1[-4] = np.pi*1
    q2 = np.array(q2)*0
    
    q1 = [0,-np.pi/8,0,-2*np.pi/3+.5*np.sin(t*0.4),0,np.pi/2,np.pi/4.,0,0]
    q2 = [np.pi/2,-np.pi/8,0,-2*np.pi/3+.05*np.sin(t*0.2),0,np.pi/2,np.pi/4.,0,0]
    # q2 = [0,np.pi/2+0.3*np.sin(t*1)*0,0,0,0,0,0.,0,0,]


    obstacle_positions[0,2] = 0.4# + 0.12*np.sin(t*0.3)
    obstacle_positions[1,2] = 0.4# + 0.12*np.sin(t*0.3)
    # obstacle_positions[2,0] = 0.5 + 0.05*np.sin(t*0.2)
    # obstacle_positions[2,2] = 0.6 + 0.05*np.sin(t*0.4)
    # obstacle_positions[3,2] += 0.04*np.sin(t*0.123)
    # obstacle_positions[4,2] += 0.04*np.sin(t*0.123)
    # obstacle_positions[:] += np.random.randn(*obstacle_positions.shape)*0.002
    
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('carried_object'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[0]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_1'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[0]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_2'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[1]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_3'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[2]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_4'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[3]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_5'),np.concatenate([np.array([1,0,0,0]),obstacle_positions[4]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_6'),np.concatenate([RollPitchYaw([0.1,0.0,0.0]).ToQuaternion().wxyz(),obstacle_positions[5]]))
    p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_7'),np.concatenate([(RotationMatrix.MakeXRotation(-np.pi/5).multiply(RotationMatrix(np.array([[1,0,0], [0,0,1],[0,-1.,0]]).T))).ToQuaternion().wxyz(),obstacle_positions[6]]))
    p.robot_1_position_port.FixValue(p_context, q2)
    p.robot_2_position_port.FixValue(p_context, q2)
    p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
    diag.ForcedPublish(d_context)
    simulator.AdvanceTo(1e-12)
    # self.visualizer.ForcedPublish(self.viz_context)
    # p.visualizer.ForcedPublish(p.viz_context)
    # pc = p.point_cloud_output_port.Eval(p_context)
    # pc2_2 = p.point_cloud_output_port_2.Eval(p_context)
    # print(np.asarray(p.camera_pose_lcm_output_port_2.Eval(p_context).pose).reshape(4,4))
    # p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
    # meshcat.SetObject("/point_cloud", pc)

    # meshcat.SetObject("/point_cloud_2", pc2_2)
    time.sleep(0.05)
    # sadfsdf
pc = p.point_cloud_output_port.Eval(p_context)
# d = p.depth_output_port.Eval(p_context)
p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
meshcat.SetObject("/point_cloud", pc)
# occluding spheres:
# panda_link_0
# none / 14
# panda_link_1
# 0,2,4 / 6
# panda_link_2
# 2,4,5 /6 
# panda_link_3
# 1 / 5
# panda_link_4
# 0,3,4,5 / 7
# panda_link_5
# 0,2,4,6,10 / 12
# panda_link_6
# 0,3 / 5 
# panda_link_7
# 0 / 2
# panda_link_8
# 0 / 1

RuntimeError: Drake could not initialize EGL by loading libEGL.so.1; please see https://drake.mit.edu/troubleshooting.html#gl-init for help.

RotationMatrix([
  [1.0, 0.0, 0.0],
  [0.0, 0.0, -1.0],
  [0.0, 1.0, 0.0],
])

In [ ]:
pc = p.point_cloud_output_port.Eval(p_context)
pc2_2 = p.point_cloud_output_port_2.Eval(p_context)
p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
meshcat.SetObject("/point_cloud", pc)

meshcat.SetObject("/point_cloud_2", pc2_2)

NameError: name 'p' is not defined

In [ ]:
from pydrake.all import PackageMap, MultibodyPlant
# PackageMap().Res
(robot,) = Parser(MultibodyPlant(0)).AddModelsFromUrl("package://drake_models/franka_description/urdf/panda_arm.urdf")

In [ ]:
obstacle_positions[0]

In [ ]:
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_1'),[1,0,0,0]+obstacle_positions[0])
p.plant.SetPositions(p.context_plant,p.plant.GetModelInstanceByName('obstacle_2'),[1,0,0,0]+obstacle_positions[1])
pc = p.point_cloud_output_port.Eval(p_context)
p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
# meshcat.SetObject("/point_cloud", pc)

In [ ]:
pc = p.point_cloud_output_port.Eval(p_context)
d = p.depth_output_port.Eval(p_context)
p.visualizer.ForcedPublish(p.visualizer.GetMyContextFromRoot(p.context))
meshcat.SetObject("/point_cloud", pc)

In [ ]:
dut = LcmImageArrayToImages()
# image_to_lcm_array.image_array_t_msg_output_port()
image_to_lcm_array_context = image_to_lcm_array.GetMyContextFromRoot(d_context)
d = p.depth_output_port.Eval(p_context)
lcm_array = image_to_lcm_array.image_array_t_msg_output_port().EvalAbstract(image_to_lcm_array_context)
context = dut.CreateDefaultContext()
dut.get_input_port().FixValue(context, lcm_array)
v = dut.get_output_port(1).Eval(context)


In [ ]:
import os
import multiprocessing
import casadi as ca
from pydrake.lcm import DrakeLcm
from pydrake.all import LcmImageArrayToImages
import time
from drake import lcmt_robot_state,lcmt_image_array, lcmt_image
from diff_co_mpc.pose_lcm import pose
from pydrake.all import DiagramBuilder, LogVectorOutput,LcmSubscriberSystem,VideoWriter,PixelType,LcmInterfaceSystem,LeafSystem,Value,Image,ImageDepth32F, PySerializer,ImageRgba8U


realsense_intrinsics = CameraInfo(
    width=640,
    height=480,
    fov_y=np.pi/5,
)

lcm = DrakeLcm()
builder = DiagramBuilder()

# plant_wrapper = builder.AddSystem(PlantDiagramWrap('plant_definition.yaml', meshcat, point_cloud_visualizer = True))
lcm_sys = builder.AddSystem(LcmInterfaceSystem(lcm))
import numpy as np
lcm_image_subscriber = builder.AddSystem(LcmSubscriberSystem.Make("simulation_depth_image",lcm =lcm,lcm_type=lcmt_image_array,use_cpp_serializer=True,wait_for_message_on_initialization_timeout = 0))
robot_1_pose_subscriber = builder.AddSystem(LcmSubscriberSystem.Make("robot_1_pose",lcm =lcm,lcm_type=lcmt_robot_state,use_cpp_serializer=False,wait_for_message_on_initialization_timeout = 0))
robot_2_pose_subscriber = builder.AddSystem(LcmSubscriberSystem.Make("robot_2_pose",lcm =lcm,lcm_type=lcmt_robot_state,use_cpp_serializer=False,wait_for_message_on_initialization_timeout = 0))
camera_pose_subscriber = builder.AddSystem(LcmSubscriberSystem.Make("camera_pose",lcm =lcm,lcm_type=lcmt_image,use_cpp_serializer=False,wait_for_message_on_initialization_timeout = 0))
lcm_to_images = builder.AddSystem(LcmImageArrayToImages())
lcm_pose_to_transform = builder.AddSystem(PoseLCMToTransform())
depth_to_point_cloud = builder.AddSystem(DepthImageToPointCloud(camera_info=realsense_intrinsics, fields= BaseField.kXYZs, pixel_type=PixelType.kDepth32F))
point_cloud_filter = builder.AddSystem(FilterRobotFromPointCloud())
svm_pipeline = builder.AddSystem(SVMPipeline())

builder.Connect(lcm_image_subscriber.get_output_port(),lcm_to_images.image_array_t_input_port())
builder.Connect(camera_pose_subscriber.get_output_port(),lcm_pose_to_transform.get_input_port())
builder.Connect(lcm_pose_to_transform.get_output_port(),depth_to_point_cloud.camera_pose_input_port())
builder.Connect(lcm_to_images.depth_image_output_port(),depth_to_point_cloud.depth_image_input_port())
builder.Connect(robot_1_pose_subscriber.get_output_port(),point_cloud_filter.robot_1_position_port)
builder.Connect(robot_2_pose_subscriber.get_output_port(),point_cloud_filter.robot_2_position_port)
builder.Connect(depth_to_point_cloud.point_cloud_output_port(),point_cloud_filter.point_cloud_input_port)
# builder

diag = builder.Build()
diagram_context = diag.CreateDefaultContext()

lcm_to_images_context = lcm_to_images.GetMyMutableContextFromRoot(diagram_context)
from pydrake.all import Simulator
# https://github.com/RobotLocomotion/drake/blob/master/bindings/pydrake/systems/test/lcm_test.py
simulator = Simulator(diag,diagram_context)
diagram_context.SetTime(time.time())
dt = 0.1
try:
    while True:
        
        t0 = time.time()

        simulator.AdvanceTo(t0 + dt)
        actual_dt = time.time() - t0
        print(diagram_context.get_time())
        
        time.sleep(dt - actual_dt)
except KeyboardInterrupt:
    pass

# lcm.unsubscribe(subscription)

In [ ]:

from pydrake.all import DrakeLcm
lcm = DrakeLcm()
state = pose()
state.pose = np.random.randn(4,4).flatten()


COMMAND_HZ = 20
lcm = DrakeLcm()

def my_handler(*args):
    # print(args[0].decode())
    msg = pose.decode(args[0])
    # print(msg.joint_name)
    print(np.asarray(msg.pose).reshape(4,4))
subscription = lcm.Subscribe("IIWA_COMMAND", my_handler)
subscription
try:
    while True:
        lcm.Publish(channel="IIWA_COMMAND", buffer = state.encode())
        lcm.HandleSubscriptions(0)
        time.sleep(0.0001)
except KeyboardInterrupt:
    pass

In [ ]:
COMMAND_HZ = 20
lcm = DrakeLcm()

def my_handler(*args):
    # print(args[0].decode())
    msg = example_t.decode(args[0])
    # print(msg.joint_name)
    print(msg.position)
subscription = lcm.Subscribe("IIWA_COMMAND", my_handler)
subscription
try:
    while True:
        lcm.Publish(channel="IIWA_COMMAND", buffer = state.encode())
        lcm.HandleSubscriptions(0)
        time.sleep(0.0001)
except KeyboardInterrupt:
    pass

# Drake LCM

In [ ]:
lcm = DrakeLcm()
subscription = lcm.Subscribe("IIWA_COMMAND", my_handler)

try:
    while True:
        lcm.HandleSubscriptions(0)
        time.sleep(0.0001)
except KeyboardInterrupt:
    pass


# ROS
TODO